### Bronze

In [0]:
%sql
USE flights_bronze;

CREATE TABLE IF NOT EXISTS airlines (
  iata_code STRING,
  airline STRING
);

CREATE TABLE IF NOT EXISTS airports (
  iata_code STRING,
  airport STRING,
  city STRING,
  state STRING,
  country STRING,
  latitude DOUBLE,
  longitude DOUBLE
);

CREATE TABLE IF NOT EXISTS flights (
  year INT,
  month INT,
  day INT,
  day_of_week INT,
  airline STRING,
  flight_number INT,
  tail_number STRING,
  origin_airport STRING,
  destination_airport STRING,
  scheduled_departure STRING,
  departure_time STRING,
  departure_delay INT,
  taxi_out  INT,
  wheels_off STRING,
  scheduled_time INT,
  elapsed_time INT,
  air_time INT,
  distance INT,
  wheels_on STRING,
  taxi_in INT,
  scheduled_arrival STRING,
  arrival_time STRING,
  arrival_delay INT,
  diverted INT,
  cancelled INT,
  cancellation_reason STRING,
  air_system_delay INT,
  security_delay INT,
  airline_delay INT,
  late_aircraft_delay INT,
  weather_delay INT
);


### Silver

In [0]:
%sql
USE flights_silver;

CREATE TABLE IF NOT EXISTS dim_airlines (
  airline_key STRING, -- klucz
  iata_code STRING,
  airline_name STRING,
  is_current BOOLEAN,
  start_date DATE,
  end_date DATE
);

CREATE TABLE IF NOT EXISTS dim_airports (
  airport_key STRING, -- klucz
  iata_code STRING,
  airport_name STRING,
  city STRING,
  state STRING,
  country STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  is_current BOOLEAN,
  start_date DATE,
  end_date DATE
);

-- flights to nasza docelowa tabela faktów, ale w warstwie silver musimy ją najpierw oczyścić 
-- -> zmiana stringów na timestampy i null handling

CREATE TABLE IF NOT EXISTS flights (
  -- data
  year INT,
  month INT,
  day INT,
  day_of_week INT,
  -- informacje o locie
  airline STRING,
  flight_number INT,
  tail_number STRING,
  origin_airport STRING,
  destination_airport STRING,
  -- czas
  scheduled_departure TIMESTAMP,
  departure_time TIMESTAMP,
  -- dane (miary) dotyczące lotu
  departure_delay INT,
  taxi_out INT,
  wheels_off TIMESTAMP, 
  scheduled_time INT,
  elapsed_time INT,
  air_time INT,
  distance INT,
  wheels_on TIMESTAMP,
  taxi_in INT,
  scheduled_arrival TIMESTAMP, 
  arrival_time TIMESTAMP,
  arrival_delay INT,
  -- czy przekierowany/odwołany
  diverted INT,
  cancelled INT,
  -- powody opóźnienia
  cancellation_reason STRING, 
  air_system_delay INT,
  security_delay INT,
  airline_delay INT,
  late_aircraft_delay INT,
  weather_delay INT
);

### Gold

In [0]:
%sql
USE flights_gold;

-- TABELA FAKTÓW
CREATE TABLE IF NOT EXISTS flights_gold.fact_flights (
    -- klucze czasowe i identyfikujące
    year INT,
    month INT,
    day INT,
    day_of_week INT,
    flight_number INT,
    tail_number STRING,
    
    -- linia lotnicza
    airline_code STRING,
    airline_name STRING,   
    -- lotnisko wylotu
    origin_code STRING,
    origin_airport_name STRING,
    origin_city STRING,
    -- lotnisko przylotu
    dest_code STRING,
    dest_airport_name STRING,
    dest_city STRING,

    -- timestampy
    scheduled_departure TIMESTAMP,
    departure_time TIMESTAMP, 
    wheels_off TIMESTAMP,     
    wheels_on TIMESTAMP,      
    scheduled_arrival TIMESTAMP,
    arrival_time TIMESTAMP,   

    -- metryki czasowe
    scheduled_time INT,       
    elapsed_time INT,         
    air_time INT,
    distance INT,
    taxi_out INT,             
    taxi_in INT,   

    -- statusy
    cancelled INT,
    cancellation_reason STRING,
    diverted INT,           
    
    -- opóźnienia i ich przyczyny
    departure_delay INT,
    arrival_delay INT,
    air_system_delay INT,
    security_delay INT,
    airline_delay INT,
    late_aircraft_delay INT,
    weather_delay INT
    
)
USING DELTA
-- partycjonowanie po roku (na razie tylko 2015) i miesiącu dla przyspieszenia filtrowania w raportach
PARTITIONED BY (year, month)

In [0]:
%sql
-- TABELA AGREGACYJNA (statystyki miesięczne)
CREATE TABLE IF NOT EXISTS flights_gold.monthly_airline_stats (
    year INT,
    month INT,
    airline_name STRING,
    
    -- główne statystyki
    total_flights LONG,
    cancelled_flights LONG,
    cancellation_rate DOUBLE, -- procent odwołanych lotów
    avg_dep_delay DOUBLE,
    avg_arr_delay DOUBLE,
    
    -- czas lotu
    avg_air_time DOUBLE,
    avg_scheduled_time DOUBLE,
    avg_elapsed_time DOUBLE,
    
    -- kołowania
    avg_taxi_out DOUBLE,      -- średni czas kołowania przy wylocie
    avg_taxi_in DOUBLE,       -- średni czas kołowania przy przylocie
    
    -- przyczyny opóźnień (średni czas)
    avg_weather_delay DOUBLE,
    avg_airline_delay DOUBLE,
    avg_security_delay DOUBLE,
    avg_system_delay DOUBLE,
    avg_late_aircraft_delay DOUBLE
)
USING DELTA